## 寶石方塊 AI - 極限效能與絕對精準版
此版本已經整合了以下最新功能：
1. **極限中心採樣 (margin=0.40)**：完全無視寶石邊緣發光特效。
2. **嚴格紅橘分離 (Strict HSV)**：提高飽和度門檻，並拉開紅橘色相緩衝區。
3. **高標準判定門檻 (Threshold=45%)**：寧可漏判也不誤判，防卡死。
4. **光影均勻化 (CLAHE) & 複合形態學 (Close+Open)**：穩定亮度與去空洞雜訊。
5. **極速操作設定 (Speed Up)**：縮短 pyautogui 延遲與等待動畫的時間。

In [ ]:
import cv2
import numpy as np
import pyautogui
import time
import random
from copy import deepcopy

# ==========================================
# 系統與安全設定 (極速版)
# ==========================================
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0.02  # 加快全域滑鼠操作延遲

# ==========================================
# 遊戲盤面座標
# ==========================================
BOARD_X = 500      
BOARD_Y = 300      
TILE_SIZE = 60     
ROWS = 8
COLS = 8

# ==========================================
# 精準 HSV 顏色範圍設定 (嚴格紅橘分離版)
# ==========================================
COLOR_RANGES = {
    'red': ([0, 180, 120], [5, 255, 255]),      # 嚴格紅色
    'orange': ([12, 150, 150], [22, 255, 255]), # 拉開與紅色的距離
    'yellow': ([26, 120, 150], [35, 255, 255]),
    'green': ([36, 100, 100], [85, 255, 255]),
    'blue': ([90, 150, 150], [130, 255, 255]),
    'purple': ([135, 100, 100], [165, 255, 255]),
    'white': ([0, 0, 150], [180, 50, 255])
}

def clamp_coords(x, y):
    min_x = BOARD_X
    max_x = BOARD_X + (COLS * TILE_SIZE) - 1
    min_y = BOARD_Y
    max_y = BOARD_Y + (ROWS * TILE_SIZE) - 1
    return max(min_x, min(x, max_x)), max(min_y, min(y, max_y))

def capture_board():
    region = (BOARD_X, BOARD_Y, COLS * TILE_SIZE, ROWS * TILE_SIZE)
    screenshot = pyautogui.screenshot(region=region)
    img_bgr = cv2.cvtColor(np.array(screenshot), cv2.COLOR_RGB2BGR)
    
    # 1. 均勻化亮度與對比 (CLAHE)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_bgr = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
    
    # 2. 輕微高斯模糊
    img_bgr = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    
    board = [['unknown' for _ in range(COLS)] for _ in range(ROWS)]
    
    for r in range(ROWS):
        for c in range(COLS):
            # 極限中心採樣 (只取正中心 20% 面積)
            margin = int(TILE_SIZE * 0.40)
            roi_hsv = img_hsv[r*TILE_SIZE+margin : (r+1)*TILE_SIZE-margin, 
                              c*TILE_SIZE+margin : (c+1)*TILE_SIZE-margin]
            roi_bgr = img_bgr[r*TILE_SIZE+margin : (r+1)*TILE_SIZE-margin, 
                              c*TILE_SIZE+margin : (c+1)*TILE_SIZE-margin]
            
            best_color = 'unknown'
            max_pixels = 0
            
            for color_name, (lower, upper) in COLOR_RANGES.items():
                lower_np = np.array(lower, dtype=np.uint8)
                upper_np = np.array(upper, dtype=np.uint8)
                mask = cv2.inRange(roi_hsv, lower_np, upper_np)
                
                # 紅色橫跨 HSV 兩端
                if color_name == 'red':
                    mask2 = cv2.inRange(roi_hsv, np.array([170, 180, 120], dtype=np.uint8), np.array([180, 255, 255], dtype=np.uint8))
                    mask = cv2.bitwise_or(mask, mask2)
                
                # 複合形態學：先閉運算填補空洞，再開運算去雜訊
                kernel_close = np.ones((5, 5), np.uint8)
                kernel_open = np.ones((3, 3), np.uint8)
                mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
                mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
                    
                pixel_count = cv2.countNonZero(mask)
                if pixel_count > max_pixels:
                    max_pixels = pixel_count
                    best_color = color_name
            
            total_area = roi_hsv.shape[0] * roi_hsv.shape[1]
            # 嚴格判定門檻：45%
            if max_pixels < (total_area * 0.45):
                best_color = 'unknown'
            
            # 特殊寶石偵測保留 (透過邊緣密度)
            gray_roi = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray_roi, 50, 150)
            edge_density = cv2.countNonZero(edges)
            
            if edge_density > 80 and best_color != 'unknown':
                best_color += "_special"
                
            board[r][c] = best_color
            
    return board

def calculate_score(board):
    score = 0
    def get_base_color(gem_name):
        return gem_name.replace('_special', '') if gem_name != 'unknown' else 'unknown'

    for r in range(ROWS):
        c = 0
        while c < COLS - 2:
            c1, c2, c3 = get_base_color(board[r][c]), get_base_color(board[r][c+1]), get_base_color(board[r][c+2])
            if c1 != 'unknown' and c1 == c2 == c3:
                length = 3
                has_special = ('_special' in board[r][c] or '_special' in board[r][c+1] or '_special' in board[r][c+2])
                while c + length < COLS and c1 == get_base_color(board[r][c+length]):
                    if '_special' in board[r][c+length]: has_special = True
                    length += 1
                base_score = {3: 100, 4: 500}.get(length, 1000)
                score += (base_score * 3) if has_special else base_score
                c += length
            else:
                c += 1
                
    for c in range(COLS):
        r = 0
        while r < ROWS - 2:
            c1, c2, c3 = get_base_color(board[r][c]), get_base_color(board[r+1][c]), get_base_color(board[r+2][c])
            if c1 != 'unknown' and c1 == c2 == c3:
                length = 3
                has_special = ('_special' in board[r][c] or '_special' in board[r+1][c] or '_special' in board[r+2][c])
                while r + length < ROWS and c1 == get_base_color(board[r+length][c]):
                    if '_special' in board[r+length][c]: has_special = True
                    length += 1
                base_score = {3: 100, 4: 500}.get(length, 1000)
                score += (base_score * 3) if has_special else base_score
                r += length
            else:
                r += 1
                
    return score

def find_all_moves(board):
    moves = []
    directions = [(0, 1), (1, 0)]
    
    for r in range(ROWS):
        for c in range(COLS):
            for dr, dc in directions:
                nr, nc = r + dr, c + dc
                if nr < ROWS and nc < COLS:
                    if board[r][c] == board[nr][nc] or board[r][c] == 'unknown' or board[nr][nc] == 'unknown':
                        continue
                        
                    temp_board = deepcopy(board)
                    temp_board[r][c], temp_board[nr][nc] = temp_board[nr][nc], temp_board[r][c]
                    
                    score = calculate_score(temp_board)
                    if score > 0:
                        depth_weight = (r + nr) * 5
                        total_score = score + depth_weight
                        moves.append((total_score, ((r, c), (nr, nc))))
                            
    moves.sort(key=lambda x: x[0], reverse=True)
    return [m[1] for m in moves]

def execute_move(move):
    (r1, c1), (r2, c2) = move
    raw_x1 = BOARD_X + c1 * TILE_SIZE + TILE_SIZE // 2
    raw_y1 = BOARD_Y + r1 * TILE_SIZE + TILE_SIZE // 2
    raw_x2 = BOARD_X + c2 * TILE_SIZE + TILE_SIZE // 2
    raw_y2 = BOARD_Y + r2 * TILE_SIZE + TILE_SIZE // 2
    
    x1, y1 = clamp_coords(raw_x1, raw_y1)
    x2, y2 = clamp_coords(raw_x2, raw_y2)
    
    pyautogui.click(x1, y1)
    time.sleep(0.05) # 極速點擊間隔
    pyautogui.click(x2, y2)
    print(f"[執行] 點擊交換 ({r1},{c1}) 與 ({r2},{c2})")

def anti_stuck():
    print("[警告] 偵測不到可消除的組合，啟動隨機點擊防卡死...")
    raw_rx = BOARD_X + random.randint(0, COLS-1) * TILE_SIZE + TILE_SIZE // 2
    raw_ry = BOARD_Y + random.randint(0, ROWS-1) * TILE_SIZE + TILE_SIZE // 2
    rx, ry = clamp_coords(raw_rx, raw_ry)
    pyautogui.click(rx, ry)
    time.sleep(0.5) # 防卡死等待稍微短一點

def is_same_board(b1, b2):
    if b1 is None or b2 is None:
        return False
    same_count = 0
    for r in range(ROWS):
        for c in range(COLS):
            c1 = b1[r][c].replace('_special', '')
            c2 = b2[r][c].replace('_special', '')
            if c1 == c2:
                same_count += 1
    return same_count >= 58

def main():
    print("=== 極限效能與絕對精準版 AI 啟動 ===")
    time.sleep(3)
    
    consecutive_stuck = 0
    previous_board = None
    failed_moves = set()

    while True:
        board = capture_board()
        
        if is_same_board(board, previous_board):
            print("[提示] 盤面未改變，剛剛的移動無效，嘗試清單下一個。")
        else:
            failed_moves = set()
            previous_board = board

        all_moves = find_all_moves(board)
        valid_moves = [m for m in all_moves if m not in failed_moves]
        
        if valid_moves:
            consecutive_stuck = 0
            best_move = valid_moves[0]
            failed_moves.add(best_move)
            execute_move(best_move)
            time.sleep(0.3) # 縮短動畫等待時間
        else:
            consecutive_stuck += 1
            anti_stuck()
            if consecutive_stuck >= 5:
                print("[中止] 連續多次無法辨識或無步可走，腳本自動停止。")
                break

if __name__ == "__main__":
    main()


### 1. 座標測量工具
請先執行此區塊取得您螢幕的正確 `BOARD_X` 與 `BOARD_Y`。

In [ ]:
import pyautogui
import time

print("【座標測量工具】啟動...")
try:
    while True:
        x, y = pyautogui.position()
        print(f"目前滑鼠座標: X: {x}, Y: {y}", end="\r")
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\n測量結束。")


### 2. AI 視覺判定除錯圖 (Visual Debugger)
執行此區塊，會截圖並印出 AI 實際看到且判定出的顏色。

In [ ]:
import cv2
import numpy as np
import pyautogui
import matplotlib.pyplot as plt

region = (BOARD_X, BOARD_Y, COLS * TILE_SIZE, ROWS * TILE_SIZE)
screenshot = pyautogui.screenshot(region=region)
img_rgb = np.array(screenshot)

board = capture_board() 

plt.figure(figsize=(10, 10))
plt.imshow(img_rgb)

for r in range(ROWS):
    for c in range(COLS):
        color_text = board[r][c]
        text_x = c * TILE_SIZE + TILE_SIZE / 2
        text_y = r * TILE_SIZE + TILE_SIZE / 2
        
        plt.text(text_x, text_y, color_text, 
                 color='white', backgroundcolor='black',
                 ha='center', va='center', fontsize=9)

plt.title("AI 視覺判定除錯圖")
plt.axis('off')
plt.show()


### 3. (選用) HSV 動態校正工具
執行後調整視窗中的拉桿，微調 HSV 數值。

In [ ]:
import cv2
import numpy as np
import pyautogui

def empty(a):
    pass

print("【HSV 動態校正工具】請在彈出的視窗調整拉桿。按下 'q' 結束並印出數值。")
region = (BOARD_X + TILE_SIZE*3, BOARD_Y + TILE_SIZE*3, TILE_SIZE*2, TILE_SIZE*2)
screenshot = pyautogui.screenshot(region=region)
img = cv2.cvtColor(np.array(screenshot), cv2.COLOR_RGB2BGR)
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

cv2.namedWindow("Trackbars")
cv2.resizeWindow("Trackbars", 640, 240)
cv2.createTrackbar("Hue Min", "Trackbars", 0, 179, empty)
cv2.createTrackbar("Hue Max", "Trackbars", 179, 179, empty)
cv2.createTrackbar("Sat Min", "Trackbars", 0, 255, empty)
cv2.createTrackbar("Sat Max", "Trackbars", 255, 255, empty)
cv2.createTrackbar("Val Min", "Trackbars", 0, 255, empty)
cv2.createTrackbar("Val Max", "Trackbars", 255, 255, empty)

while True:
    h_min = cv2.getTrackbarPos("Hue Min", "Trackbars")
    h_max = cv2.getTrackbarPos("Hue Max", "Trackbars")
    s_min = cv2.getTrackbarPos("Sat Min", "Trackbars")
    s_max = cv2.getTrackbarPos("Sat Max", "Trackbars")
    v_min = cv2.getTrackbarPos("Val Min", "Trackbars")
    v_max = cv2.getTrackbarPos("Val Max", "Trackbars")
    
    lower = np.array([h_min, s_min, v_min])
    upper = np.array([h_max, s_max, v_max])
    mask = cv2.inRange(img_hsv, lower, upper)
    result = cv2.bitwise_and(img, img, mask=mask)
    
    cv2.imshow("Original", img)
    cv2.imshow("Mask", mask)
    cv2.imshow("Result", result)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print(f"最佳範圍 -> Lower: [{h_min}, {s_min}, {v_min}], Upper: [{h_max}, {s_max}, {v_max}]")
        break

cv2.destroyAllWindows()
